> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Uncertainty in a computation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/main/docs/user_guide/01_uncertainty_in_a_computation.ipynb)

A computation can combine a known value with an uncertain quantity.
> 1. A **value** holds known data for a computation.
> 2. A **distribution** describes uncertainty about a value.
> 3. A **function** computes an output from inputs.

ProbPipe lets a function accept either values or distributions where it expects numeric data.
This chapter shows the three working together to compute the mean of five measurements.

After the example, choose a chapter for your next task.

In [1]:
# On Google Colab, install ProbPipe; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab's preinstalled PyMC 5 does not work with ArviZ 1.x, so the install upgrades PyMC.
    %pip install --quiet "probpipe-core[pymc] @ git+https://github.com/TARPS-group/prob-pipe.git@main"

In [2]:
import jax
import jax.numpy as jnp

from probpipe import Normal, NumericArray, function, mean, workflow_run

## A value, a distribution, and a function

Suppose our first four measurements are `[4, 5, 6, 5]`.
We want the mean after adding an additional measurement.
We store the first four as a `NumericArray`, which holds numeric data, and write a function that includes the last measurement in the average:

In [3]:
measurements = NumericArray(jnp.array([4.0, 5.0, 6.0, 5.0]), label="measurements")


@function
def sample_mean(values: jax.Array, last: jax.Array) -> jax.Array:
    return (jnp.sum(values) + last) / (values.size + 1)


known_mean = sample_mean(measurements, 5.0)
print("first measurements:", measurements.raw)
print("mean with an additional measurement of 5:", known_mean.raw)

first measurements: [4. 5. 6. 5.]
mean with an additional measurement of 5: 5.0


`@function` lets ProbPipe apply the computation to values or distributions.
Called with known values, `sample_mean` returns one value, here 5.
The result is a `NumericArray`, whose `.raw` gives the number for use in other Python code.

Now suppose the fifth measurement is uncertain.
We describe it by a normal distribution with mean 5 and standard deviation 1:

In [4]:
last = Normal("last", 5.0, 1.0)
print("distribution of the fifth measurement:", last)

distribution of the fifth measurement: Normal(last)


`Normal` takes the output name `"last"`, then the mean and standard deviation.
The output names the quantity that a draw represents.

We pass this distribution to the same function, keeping the first four measurements fixed:

In [5]:
with workflow_run(seed=0):
    uncertain_mean = sample_mean(measurements, last)

print("distribution of the sample mean:", uncertain_mean)
print("expected sample mean:", mean(uncertain_mean).raw)

distribution of the sample mean: sample_mean(measurements, last ~ Normal)
expected sample mean: 4.9897914


The result describes uncertainty in the five-measurement average, keeping the first four measurements fixed.
By default, ProbPipe approximates this distribution by drawing 256 possible fifth measurements and computing an average for each draw.
You can change this evaluation budget, as the functions chapter explains.
It represents the resulting averages as an `EmpiricalDistribution`.
This is a distribution on finitely many values.
The seed in `workflow_run` makes this computation reproducible.

Under this model, the average has expectation 5 and standard deviation 0.2.
Changing the fifth measurement by 1 changes the average by 1/5, so its standard deviation of 1 becomes 0.2 in the average.
The mean printed above is close to 5 because ProbPipe approximates the distribution using finitely many random draws.

| Object | Role in this computation |
|---|---|
| `measurements` | An array of four known measurements passed to every evaluation |
| `last` | A distribution describing uncertainty in the fifth measurement |
| `sample_mean` | A function that averages the five measurements |
| `known_mean` | A value returned when all five measurements are known |
| `uncertain_mean` | A distribution returned when the fifth measurement is uncertain |

Values, distributions, and functions are all **terms** in ProbPipe.
An **operation**, such as `mean`, computes a result from terms.
Here `sample_mean` averages measurements, while `mean(uncertain_mean)` computes the expectation of the resulting distribution.
[Distributions](02_distributions.ipynb) covers intervals and sampling from a distribution.
[Propagating uncertainty through functions](03_functions_and_lifting.ipynb) explains how to control the accuracy of a lifted computation.

## Where to go next

- [Distributions](02_distributions.ipynb): construct distributions, draw values, and compute summaries.
- [Propagating uncertainty through functions](03_functions_and_lifting.ipynb): propagate uncertainty through your own functions, control the number of draws, and declare the inputs and the output that a function accepts.
- [Your data: values, records, and batches](04_values_and_records.ipynb): pass data into computations and read the results.

[Tracking and reproducing results](09_how_results_are_computed.ipynb) explains the labels that name each result, reproducible draws, and the record of a result's computation, called its provenance.

## See also

- [Values and records](../api/values.md): the API reference for `NumericArray` and the other values.
- [Functions](../api/functions.md): `function` and the rules for calling a function on values, distributions, and batches.
- [Workflows and reproducibility](../api/workflows.md): `workflow_run`.
- [Tutorial 1](../tutorials/01_first_analysis.ipynb) follows a complete analysis from a prior to predictions.